In [2]:
# =============================================================================
# ATIVIDADE DIDÁTICA: "CONECTANDO LUGARES"
# Rotas, distâncias e desigualdades territoriais com Folium
# Ensino de Geografia com Python — Licenciatura em Geografia
# =============================================================================
# COMO RODAR NO GOOGLE COLAB:
#   1. Acesse https://colab.research.google.com
#   2. Clique em "Novo notebook"
#   3. Cole este código célula por célula (separadas pelos blocos # ── CÉLULA)
#   4. Execute cada célula com Shift+Enter ou clicando no botão ▶
#
# ATENÇÃO: o Folium NÃO vem pré-instalado no Colab.
#   Execute a Célula 1 para instalar antes de tudo.
# =============================================================================

# -----------------------------------------------------------------------------
# CÉLULA 1 — Instalação e importação das bibliotecas
# -----------------------------------------------------------------------------
# folium: criação de mapas interativos no navegador (baseado em Leaflet.js)
# pandas: manipulação de tabelas de dados
# math: cálculo de distâncias geográficas (fórmula de Haversine)

# Instalação do folium (necessária no Colab a cada nova sessão)
# Execute esta célula primeiro e aguarde a conclusão antes de prosseguir.
import subprocess
subprocess.run(['pip', 'install', 'folium', '--quiet'], check=True)

import folium
from folium import plugins
from folium.plugins import MarkerCluster, MiniMap, MeasureControl
import pandas as pd
import math
import json

print("✅ Bibliotecas carregadas com sucesso!")
print(f"   Folium versão: {folium.__version__}")

# -----------------------------------------------------------------------------
# CÉLULA 2 — Dados: equipamentos públicos de uma cidade fictícia
# -----------------------------------------------------------------------------
# Trabalhamos com coordenadas de uma cidade fictícia chamada "Vila Esperança",
# inspirada em municípios brasileiros de médio porte.
# Os alunos podem substituir esses dados pelas coordenadas reais de sua cidade,
# obtidas gratuitamente no Google Maps (clique com botão direito → "O que há aqui?")
# ou no OpenStreetMap (https://www.openstreetmap.org).
#
# Referência pedagógica: Callai (2005) — partir do lugar vivido para
# compreender o espaço geográfico mais amplo.

# Ponto central da cidade (praça principal)
CENTRO = [-23.5505, -46.6333]  # substitua pelas coordenadas do centro da sua cidade

# ── ESCOLAS MUNICIPAIS ────────────────────────────────────────────────────────
escolas = pd.DataFrame({
    'nome': [
        'EMEF João Paulo II',
        'EMEF Monteiro Lobato',
        'EMEF Zumbi dos Palmares',
        'EMEF Santos Dumont',
        'EMEF Maria Quitéria',
    ],
    'lat': [-23.5420, -23.5610, -23.5350, -23.5680, -23.5530],
    'lon': [-46.6250, -46.6410, -46.6480, -46.6200, -46.6550],
    'bairro': ['Centro', 'Jardim Sul', 'Bela Vista', 'Parque Norte', 'Vila Nova'],
    'vagas': [480, 320, 290, 410, 350],
    'ied': [0.72, 0.58, 0.51, 0.69, 0.55],  # Índice de Estrutura e Docência (0-1)
})

# ── UNIDADES DE SAÚDE ─────────────────────────────────────────────────────────
saude = pd.DataFrame({
    'nome': [
        'UBS Centro',
        'UBS Jardim Sul',
        'Hospital Municipal',
        'UPA 24h Norte',
        'UBS Vila Nova',
    ],
    'lat': [-23.5480, -23.5640, -23.5500, -23.5300, -23.5570],
    'lon': [-46.6310, -46.6440, -46.6280, -46.6350, -46.6500],
    'tipo': ['UBS', 'UBS', 'Hospital', 'UPA', 'UBS'],
    'bairro': ['Centro', 'Jardim Sul', 'Centro', 'Parque Norte', 'Vila Nova'],
    'atend_dia': [180, 120, 450, 310, 140],
})

# ── PARQUES E ÁREAS VERDES ───────────────────────────────────────────────────
parques = pd.DataFrame({
    'nome': [
        'Parque da Cidade',
        'Praça da República',
        'Parque Linear Sul',
        'Praça dos Imigrantes',
    ],
    'lat': [-23.5460, -23.5510, -23.5650, -23.5400],
    'lon': [-46.6270, -46.6330, -46.6420, -46.6180],
    'area_m2': [45000, 8500, 22000, 5200],
    'bairro': ['Centro', 'Centro', 'Jardim Sul', 'Bela Vista'],
})

# ── RESIDÊNCIAS DOS ALUNOS (exemplo) ─────────────────────────────────────────
# Na atividade real, cada aluno informa o bairro/endereço aproximado.
# Nunca usar o endereço exato — usar o centroide do bairro para preservar privacidade.
alunos = pd.DataFrame({
    'aluno': ['Aluno A', 'Aluno B', 'Aluno C', 'Aluno D', 'Aluno E'],
    'bairro': ['Centro', 'Jardim Sul', 'Bela Vista', 'Parque Norte', 'Vila Nova'],
    'lat': [-23.5490, -23.5620, -23.5370, -23.5660, -23.5550],
    'lon': [-46.6300, -46.6430, -46.6470, -46.6220, -46.6520],
    'escola_ref': [0, 1, 2, 3, 4],  # índice da escola mais próxima (calculado abaixo)
})

print("✅ Dados carregados!")
print(f"   {len(escolas)} escolas | {len(saude)} unidades de saúde | "
      f"{len(parques)} parques | {len(alunos)} alunos")

# -----------------------------------------------------------------------------
# CÉLULA 3 — Função de distância geográfica (fórmula de Haversine)
# -----------------------------------------------------------------------------
# A fórmula de Haversine calcula a distância entre dois pontos na superfície
# da Terra a partir de suas coordenadas geográficas (latitude e longitude).
# O resultado é dado em quilômetros.
# É mais precisa que o cálculo euclidiano porque considera a curvatura da Terra.

def haversine(lat1, lon1, lat2, lon2):
    """
    Calcula a distância em km entre dois pontos geográficos.
    Parâmetros:
        lat1, lon1: coordenadas do ponto de origem
        lat2, lon2: coordenadas do ponto de destino
    Retorna:
        distância em quilômetros (float)
    """
    R = 6371  # raio médio da Terra em km
    d_lat = math.radians(lat2 - lat1)
    d_lon = math.radians(lon2 - lon1)
    a = (math.sin(d_lat / 2) ** 2 +
         math.cos(math.radians(lat1)) *
         math.cos(math.radians(lat2)) *
         math.sin(d_lon / 2) ** 2)
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

# Calcula distância de cada aluno até cada escola e encontra a mais próxima
print("\n📏 Distância de cada aluno até a escola mais próxima:")
print("-" * 55)

distancias_alunos = []
for _, aluno in alunos.iterrows():
    dists = []
    for _, escola in escolas.iterrows():
        d = haversine(aluno['lat'], aluno['lon'], escola['lat'], escola['lon'])
        dists.append((escola['nome'], round(d, 2)))
    mais_proxima = min(dists, key=lambda x: x[1])
    distancias_alunos.append({
        'aluno': aluno['aluno'],
        'bairro': aluno['bairro'],
        'escola_proxima': mais_proxima[0],
        'distancia_km': mais_proxima[1],
        'tempo_caminhada_min': round(mais_proxima[1] / 5 * 60)  # ~5 km/h
    })
    print(f"  {aluno['aluno']} ({aluno['bairro']:12s}) → "
          f"{mais_proxima[0][:22]:22s} | {mais_proxima[1]:.2f} km | "
          f"~{round(mais_proxima[1] / 5 * 60)} min a pé")

df_distancias = pd.DataFrame(distancias_alunos)

# -----------------------------------------------------------------------------
# CÉLULA 4 — MAPA 1: Mapa base com todos os equipamentos públicos
# -----------------------------------------------------------------------------
# O Folium cria mapas interativos em HTML — o aluno pode dar zoom,
# clicar nos marcadores e ver as informações em popups.
# tiles='CartoDB positron' usa um mapa de fundo minimalista em tons de cinza,
# que destaca as marcações coloridas sem poluição visual.

mapa1 = folium.Map(
    location=CENTRO,
    zoom_start=14,
    tiles='CartoDB positron'
)

# Adiciona mini-mapa de contexto no canto inferior direito
MiniMap(toggle_display=True).add_to(mapa1)

# Adiciona ferramenta de medição de distâncias (o aluno pode usar diretamente no mapa)
MeasureControl(
    position='topright',
    primary_length_unit='kilometers',
    secondary_length_unit='meters'
).add_to(mapa1)

# ── Marcadores das escolas ────────────────────────────────────────────────────
grupo_escolas = folium.FeatureGroup(name='🏫 Escolas Municipais')
for _, row in escolas.iterrows():
    popup_html = f"""
    <div style="font-family:Arial;min-width:200px">
      <h4 style="color:#1565C0;margin:0 0 8px 0">🏫 {row['nome']}</h4>
      <b>Bairro:</b> {row['bairro']}<br>
      <b>Vagas:</b> {row['vagas']}<br>
      <b>Índice de Estrutura:</b> {row['ied']:.2f}<br>
      <small style="color:#888">Fonte: Inep / Censo Escolar</small>
    </div>
    """
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=row['nome'],
        icon=folium.Icon(color='blue', icon='graduation-cap', prefix='fa')
    ).add_to(grupo_escolas)
grupo_escolas.add_to(mapa1)

# ── Marcadores das unidades de saúde ─────────────────────────────────────────
grupo_saude = folium.FeatureGroup(name='🏥 Unidades de Saúde')
cores_saude = {'UBS': 'green', 'Hospital': 'red', 'UPA': 'orange'}
icones_saude = {'UBS': 'plus', 'Hospital': 'hospital-o', 'UPA': 'ambulance'}
for _, row in saude.iterrows():
    popup_html = f"""
    <div style="font-family:Arial;min-width:200px">
      <h4 style="color:#2E7D32;margin:0 0 8px 0">🏥 {row['nome']}</h4>
      <b>Tipo:</b> {row['tipo']}<br>
      <b>Bairro:</b> {row['bairro']}<br>
      <b>Atendimentos/dia:</b> {row['atend_dia']}<br>
      <small style="color:#888">Fonte: CNES / DataSUS</small>
    </div>
    """
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=f"{row['tipo']}: {row['nome']}",
        icon=folium.Icon(
            color=cores_saude.get(row['tipo'], 'green'),
            icon=icones_saude.get(row['tipo'], 'plus'),
            prefix='fa'
        )
    ).add_to(grupo_saude)
grupo_saude.add_to(mapa1)

# ── Marcadores dos parques ────────────────────────────────────────────────────
grupo_parques = folium.FeatureGroup(name='🌳 Parques e Áreas Verdes')
for _, row in parques.iterrows():
    popup_html = f"""
    <div style="font-family:Arial;min-width:200px">
      <h4 style="color:#388E3C;margin:0 0 8px 0">🌳 {row['nome']}</h4>
      <b>Bairro:</b> {row['bairro']}<br>
      <b>Área:</b> {row['area_m2']:,} m²<br>
      <small style="color:#888">Fonte: Prefeitura Municipal</small>
    </div>
    """
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=max(8, row['area_m2'] / 5000),  # raio proporcional à área
        color='darkgreen',
        fill=True,
        fill_color='#4CAF50',
        fill_opacity=0.5,
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=f"🌳 {row['nome']} ({row['area_m2']:,} m²)"
    ).add_to(grupo_parques)
grupo_parques.add_to(mapa1)

# Controle de camadas (liga/desliga cada grupo de marcadores)
folium.LayerControl(collapsed=False).add_to(mapa1)

# Salva o mapa como arquivo HTML interativo
mapa1.save('mapa1_equipamentos_publicos.html')
print("✅ Mapa 1 salvo como 'mapa1_equipamentos_publicos.html'")
print("   Abra o arquivo no navegador para interagir com o mapa.")

# -----------------------------------------------------------------------------
# CÉLULA 5 — MAPA 2: Rotas dos alunos até as escolas
# -----------------------------------------------------------------------------
# Desenhamos linhas retas (voo de pássaro) entre cada aluno e sua escola
# mais próxima. Na atividade real, essas linhas podem ser substituídas por
# rotas reais de rua usando a API do OpenRouteService (gratuita).

mapa2 = folium.Map(location=CENTRO, zoom_start=14, tiles='CartoDB positron')
MiniMap(toggle_display=True).add_to(mapa2)
MeasureControl(position='topright', primary_length_unit='kilometers').add_to(mapa2)

# Paleta de cores para diferenciar os alunos
cores_alunos = ['#E53935', '#8E24AA', '#1E88E5', '#00897B', '#F4511E']

for i, (_, aluno) in enumerate(alunos.iterrows()):
    escola_idx = aluno['escola_ref']
    escola_row = escolas.iloc[escola_idx]
    dist = df_distancias.iloc[i]['distancia_km']
    tempo = df_distancias.iloc[i]['tempo_caminhada_min']
    cor = cores_alunos[i]

    # Marcador do aluno (casa)
    folium.Marker(
        location=[aluno['lat'], aluno['lon']],
        tooltip=f"🏠 {aluno['aluno']} — {aluno['bairro']}",
        popup=folium.Popup(
            f"<b>{aluno['aluno']}</b><br>Bairro: {aluno['bairro']}<br>"
            f"Distância até a escola: {dist:.2f} km<br>"
            f"Tempo a pé: ~{tempo} min",
            max_width=220
        ),
        icon=folium.Icon(color='gray', icon='home', prefix='fa')
    ).add_to(mapa2)

    # Marcador da escola de referência
    folium.Marker(
        location=[escola_row['lat'], escola_row['lon']],
        tooltip=f"🏫 {escola_row['nome']}",
        icon=folium.Icon(color='blue', icon='graduation-cap', prefix='fa')
    ).add_to(mapa2)

    # Linha de rota (tracejada) entre casa e escola
    folium.PolyLine(
        locations=[
            [aluno['lat'], aluno['lon']],
            [escola_row['lat'], escola_row['lon']]
        ],
        color=cor,
        weight=3,
        opacity=0.8,
        dash_array='8 4',  # linha tracejada
        tooltip=f"{aluno['aluno']} → {escola_row['nome']}: {dist:.2f} km"
    ).add_to(mapa2)

    # Ponto médio da linha com etiqueta de distância
    lat_mid = (aluno['lat'] + escola_row['lat']) / 2
    lon_mid = (aluno['lon'] + escola_row['lon']) / 2
    folium.Marker(
        location=[lat_mid, lon_mid],
        icon=folium.DivIcon(
            html=f'<div style="font-size:10px;font-weight:bold;'
                 f'color:{cor};background:white;padding:2px 5px;'
                 f'border-radius:4px;border:1px solid {cor};'
                 f'white-space:nowrap">{dist:.2f} km</div>',
            icon_size=(70, 20),
            icon_anchor=(35, 10)
        )
    ).add_to(mapa2)

mapa2.save('mapa2_rotas_alunos_escolas.html')
print("✅ Mapa 2 salvo como 'mapa2_rotas_alunos_escolas.html'")

# -----------------------------------------------------------------------------
# CÉLULA 6 — MAPA 3: Mapa de calor — densidade de equipamentos públicos
# -----------------------------------------------------------------------------
# O mapa de calor (heatmap) mostra visualmente onde os equipamentos públicos
# estão mais concentrados — e onde há "vazios de serviços".
# Áreas quentes (vermelho) têm mais equipamentos; áreas frias (azul), menos.
# Esse tipo de mapa é amplamente usado em análises de vulnerabilidade territorial.

from folium.plugins import HeatMap

mapa3 = folium.Map(location=CENTRO, zoom_start=14, tiles='CartoDB dark_matter')
MiniMap(toggle_display=True, tile_layer='CartoDB positron').add_to(mapa3)

# Reunimos todos os equipamentos em uma lista de coordenadas
todos_equipamentos = []
for _, row in escolas.iterrows():
    todos_equipamentos.append([row['lat'], row['lon'], 1.0])
for _, row in saude.iterrows():
    # Hospitais e UPAs têm peso maior (mais impacto territorial)
    peso = 2.0 if row['tipo'] in ['Hospital', 'UPA'] else 1.0
    todos_equipamentos.append([row['lat'], row['lon'], peso])
for _, row in parques.iterrows():
    # Parques têm peso proporcional à área
    peso = min(2.0, row['area_m2'] / 20000)
    todos_equipamentos.append([row['lat'], row['lon'], peso])

# Parâmetros do heatmap:
# radius: raio de influência de cada ponto (em pixels)
# blur: suavização da gradiente
# max_zoom: zoom máximo em que o heatmap é visível
HeatMap(
    todos_equipamentos,
    radius=35,
    blur=20,
    max_zoom=16,
    gradient={0.2: 'blue', 0.5: 'yellow', 0.8: 'orange', 1.0: 'red'}
).add_to(mapa3)

# Marcamos também as residências dos alunos para contextualizar
for i, (_, aluno) in enumerate(alunos.iterrows()):
    folium.CircleMarker(
        location=[aluno['lat'], aluno['lon']],
        radius=7,
        color='white',
        fill=True,
        fill_color=cores_alunos[i],
        fill_opacity=0.9,
        tooltip=f"🏠 {aluno['aluno']} — {aluno['bairro']}"
    ).add_to(mapa3)

# Título em overlay
titulo_html = """
<div style="position:fixed;top:12px;left:50%;transform:translateX(-50%);
     z-index:9999;background:rgba(0,0,0,0.75);color:white;
     padding:10px 20px;border-radius:8px;font-family:Arial;
     font-size:14px;font-weight:bold;text-align:center;
     box-shadow:0 2px 8px rgba(0,0,0,0.5)">
  🔥 Densidade de Equipamentos Públicos — Vila Esperança<br>
  <span style="font-size:11px;font-weight:normal;opacity:0.8">
  Vermelho = maior concentração · Azul = menor concentração
  </span>
</div>
"""
mapa3.get_root().html.add_child(folium.Element(titulo_html))

mapa3.save('mapa3_heatmap_equipamentos.html')
print("✅ Mapa 3 salvo como 'mapa3_heatmap_equipamentos.html'")

# -----------------------------------------------------------------------------
# CÉLULA 7 — ANÁLISE: desigualdades territoriais e perguntas para debate
# -----------------------------------------------------------------------------

print("=" * 65)
print("  ANÁLISE DE ACESSO AOS EQUIPAMENTOS PÚBLICOS")
print("=" * 65)

print("\n📏 DISTÂNCIAS CASA → ESCOLA (km):")
print("-" * 55)
for _, row in df_distancias.iterrows():
    barra = '█' * int(row['distancia_km'] * 10)
    print(f"  {row['aluno']} ({row['bairro']:12s}): {row['distancia_km']:.2f} km  {barra}")

media_dist = df_distancias['distancia_km'].mean()
max_dist   = df_distancias['distancia_km'].max()
min_dist   = df_distancias['distancia_km'].min()
print(f"\n  Distância média:  {media_dist:.2f} km")
print(f"  Distância máxima: {max_dist:.2f} km ({df_distancias.loc[df_distancias['distancia_km'].idxmax(), 'aluno']})")
print(f"  Distância mínima: {min_dist:.2f} km ({df_distancias.loc[df_distancias['distancia_km'].idxmin(), 'aluno']})")
print(f"  Razão máx/mín:    {max_dist/min_dist:.1f}x")

# Equipamentos por bairro
print("\n\n🏘️  EQUIPAMENTOS PÚBLICOS POR BAIRRO:")
print("-" * 55)
bairros = ['Centro', 'Jardim Sul', 'Bela Vista', 'Parque Norte', 'Vila Nova']
for b in bairros:
    n_esc  = len(escolas[escolas['bairro'] == b])
    n_sau  = len(saude[saude['bairro'] == b])
    n_par  = len(parques[parques['bairro'] == b])
    total  = n_esc + n_sau + n_par
    barra  = '█' * total + '░' * (6 - total)
    print(f"  {b:15s}: {barra} | 🏫 {n_esc} escola(s)  🏥 {n_sau} saúde  🌳 {n_par} parque(s)")

print("\n" + "=" * 65)
print("  PERGUNTAS PARA DEBATE EM SALA DE AULA")
print("=" * 65)
perguntas = [
    "1. Observe o mapa de calor: quais bairros têm mais equipamentos\n"
    "   públicos? Isso coincide com as áreas de maior renda?",
    "2. Compare as distâncias dos alunos até a escola.\n"
    "   Essa diferença é justa? Quem é mais afetado?",
    "3. Um aluno que mora a 2 km da escola leva quanto tempo\n"
    "   a pé? E de ônibus? E se não houver transporte público?",
    "4. O que significa 'direito à cidade'? Os moradores de todos\n"
    "   os bairros têm o mesmo acesso aos equipamentos públicos?",
    "5. Se você fosse gestor municipal, onde construiria a próxima\n"
    "   escola ou UBS? Justifique com base nos mapas produzidos.",
]
for p in perguntas:
    print(f"\n{p}")

print("\n" + "=" * 65)
print("  FONTES E DADOS ABERTOS UTILIZADOS")
print("=" * 65)
print("Escolas:  Inep. Censo Escolar 2023. https://inep.gov.br/censo-escolar")
print("Saúde:    DataSUS. CNES 2023. https://cnes.datasus.gov.br")
print("Parques:  Prefeitura Municipal (dados fictícios para fins didáticos)")
print("Mapas:    OpenStreetMap. https://www.openstreetmap.org")
print("Método:   Haversine (distância geodésica)")

# =============================================================================
# FIM DA ATIVIDADE
# =============================================================================
# Arquivos gerados (visíveis no painel de arquivos do Colab, à esquerda):
#   • mapa1_equipamentos_publicos.html  — mapa interativo com camadas
#   • mapa2_rotas_alunos_escolas.html   — rotas dos alunos até as escolas
#   • mapa3_heatmap_equipamentos.html   — mapa de calor da cobertura territorial
#
# Para visualizar: baixe os arquivos .html e abra no navegador,
# ou use o código abaixo no Colab para exibir embutido:
#
#   from IPython.display import IFrame
#   IFrame('mapa1_equipamentos_publicos.html', width=800, height=500)
#
# Para salvar no Drive: from google.colab import drive; drive.mount('/content/drive')
# =============================================================================

✅ Bibliotecas carregadas com sucesso!
   Folium versão: 0.20.0
✅ Dados carregados!
   5 escolas | 5 unidades de saúde | 4 parques | 5 alunos

📏 Distância de cada aluno até a escola mais próxima:
-------------------------------------------------------
  Aluno A (Centro      ) → EMEF João Paulo II     | 0.93 km | ~11 min a pé
  Aluno B (Jardim Sul  ) → EMEF Monteiro Lobato   | 0.23 km | ~3 min a pé
  Aluno C (Bela Vista  ) → EMEF Zumbi dos Palmare | 0.24 km | ~3 min a pé
  Aluno D (Parque Norte) → EMEF Santos Dumont     | 0.30 km | ~4 min a pé
  Aluno E (Vila Nova   ) → EMEF Maria Quitéria    | 0.38 km | ~5 min a pé
✅ Mapa 1 salvo como 'mapa1_equipamentos_publicos.html'
   Abra o arquivo no navegador para interagir com o mapa.
✅ Mapa 2 salvo como 'mapa2_rotas_alunos_escolas.html'
✅ Mapa 3 salvo como 'mapa3_heatmap_equipamentos.html'
  ANÁLISE DE ACESSO AOS EQUIPAMENTOS PÚBLICOS

📏 DISTÂNCIAS CASA → ESCOLA (km):
-------------------------------------------------------
  Aluno A (Centro    